# 09 — Hybrid Scoring & Ablation

## Objective
Combine validated component scores after rank normalization and select weights using validation PR-AUC.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Recreate validated components

In [ ]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); 
train,val,test,_=chronological_split(fraud); 
b=HistoryFeatureBuilder().fit(train); 
Xtr,Xv,Xt=map(b.transform,[train,val,test]);
cols=feature_columns(Xtr); 
pre=Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]);

A=pre.fit_transform(Xtr[cols]); B=pre.transform(Xv[cols]); 
iso,lof=fit_models(A,lof_sample=30000); 
Xv["iforest_score"],Xv["lof_score"]=model_scores((iso,lof),B); 

Ev,svd,Etr=graph_incidence_embedding(train,val,n_components=12); 

Xv["graph_score"]=np.linalg.norm(Ev-Etr.mean(axis=0),axis=1); 
Xv["stat_score"]=np.maximum(abs(robust_z(Xv.purchase_value)),abs(robust_z(Xv.account_age_hours)))

## 2. Ablation table

In [ ]:
from scipy.stats import rankdata
components=["stat_score","iforest_score","lof_score","graph_score"]
for c in components: Xv[c+"_rank"]=rankdata(Xv[c])/len(Xv)
candidates={"ML":(0,.65,.35,0),"ML+Graph":(0,.50,.25,.25),"Stats+ML":(.20,.50,.30,0),"Full":(.15,.45,.20,.20)}
rows=[]
for name,w in candidates.items():
 s=sum(a*Xv[c+"_rank"].values for a,c in zip(w,components));
 m=ranking_metrics(Xv['class'],s); 
 m["variant"]=name; rows.append(m)


display(pd.DataFrame(rows).sort_values("pr_auc",ascending=False)); 
px.bar(pd.DataFrame(rows),x="variant",y="pr_auc",title="Ablation: validation PR-AUC").show()

,pr_auc,roc_auc,precision_at_50,recall_at_50,precision_at_100,recall_at_100,precision_at_500,recall_at_500,variant
1,0.052571,0.495643,0.18,0.008654,0.19,0.018269,0.086,0.041346,ML+Graph
3,0.050781,0.493784,0.26,0.012500,0.19,0.018269,0.072,0.034615,Full
2,0.044729,0.488917,0.06,0.002885,0.05,0.004808,0.040,0.019231,Stats+ML
0,0.044355,0.487329,0.04,0.001923,0.05,0.004808,0.036,0.017308,ML


## 3. Freeze hybrid configuration

In [4]:
best=max(rows,key=lambda r:r["pr_auc"])["variant"]; 
weights=candidates[best]; 

Xv["hybrid_score"]=sum(w*Xv[c+"_rank"] for w,c in zip(weights,components)); 

display(Markdown(f"### Frozen candidate: **{best}**")); 

save_json({"variant":best,"weights":dict(zip(components,weights))},ART/"hybrid_config.json"); 
Xv.sort_values("hybrid_score",ascending=False).head(1000).to_parquet(ART/"validation_ranked.parquet")

### Frozen candidate: **ML+Graph**